# Follow-up experiments

**Paper:** *Where the Dimension Hides: The Approximation Constant of Kolmogorov–Arnold Networks*
**Author:** Haydar Kılıç · ORCID 0000-0002-2551-3772

Section 6.3 of the manuscript names three experiments that would strengthen it. This notebook
runs all three. It is self-contained: nothing is imported from the main pipeline notebook.

| # | Experiment | What it fixes | Cost (A100) |
|---|---|---|---|
| **E3** | Re-specify P2 against a genuine K–A topology | P2 was tested against a benchmark that does not apply to `FullKAN`; moves it from *untested* to *tested* | 3–6 GPU-h |
| **E1** | Decouple capacity from sample size | The two probes were measured in different statistical regimes; this is the change most likely to make the headline exponent measurable | 12–20 GPU-h |
| **E2** | Alternating exact solve for the two-layer family | Gives a *certified attainable* error for `KANet`, so the optimisation deficit can be demonstrated there too | hours |

They are ordered by cost, cheapest first, so a partial run still yields a usable result.

---

### One correction to how E2 was described earlier

E2 does **not** produce a sandwich inequality for the two-layer family, and the notebook does
not claim one. The obstacle is not numerical conditioning; it is structural. The outer layer is
linear in its parameters only *when the inner map is held fixed*, whereas a trained `KANet`
optimises both. A closed-form outer solve therefore bounds only the models sharing that
particular inner map — not the trained model. Proposition 2 cannot be extended to this family.

What E2 *can* give is a **certified upper bound on attainable error**: alternating between an
exact outer solve and gradient steps on the inner map produces a monotonically decreasing
sequence. If AdamW ends up materially above it, the optimisation deficit is demonstrated for
the two-layer topology as well — which Section 5.2 currently cannot do.

### Scope note

Every result here is **new** and is not part of the pre-registered study. E3 and E1 must be
reported as pre-specified-in-the-paper-but-run-afterwards; E2 is exploratory. None of them may
be used to revise the original verdict, only to extend it.

## 0. Setup

Set `SCALE` to `"smoke"` first and confirm the whole notebook runs on CPU in a few minutes.
Only then switch to `"full"` on an accelerator.

In [ ]:
import os, json, math, time, random, itertools
from dataclasses import dataclass, field, asdict
from typing import List

import numpy as np
import pandas as pd
from scipy import stats
import torch
import torch.nn as nn
import torch.nn.functional as F

SCALE = "full"           # "smoke" (CPU, minutes) | "trim" (GPU, ~30 min) | "full" (GPU, 7-10 h)

# ---- Persistent output ---------------------------------------------------------------
# Colab's /content is wiped when the runtime is recycled, which would destroy the CSVs
# that make E1, E2 and E3 resumable. Write to Drive when it is available.
REQUIRE_PERSISTENT = True    # refuse to run if results would land on ephemeral storage

def _resolve_out():
    """Resolve the output directory, and refuse to proceed if it is not persistent.

    Colab wipes /content when the runtime is recycled. An earlier version of this notebook
    fell back to /content silently when Drive was unavailable, and a completed run was lost
    that way. It now fails loudly instead.
    """
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        d = "/content/drive/MyDrive/KAN_JMLR/followup_outputs"
        os.makedirs(d, exist_ok=True)
        probe = os.path.join(d, ".write_test")
        with open(probe, "w") as f: f.write("ok")
        os.remove(probe)
        print(f"[storage] persistent: {d}")
        return d
    except ImportError:
        d = os.path.abspath("./followup_outputs")
        print(f"[storage] not on Colab; writing locally to {d}")
        return d
    except Exception as e:
        msg = ("Google Drive could not be mounted (" + type(e).__name__ + "). Results would "
               "be written to /content, which Colab deletes when the runtime is recycled -- "
               "a completed run has already been lost this way.\n"
               "Either mount Drive and re-run this cell, or set REQUIRE_PERSISTENT = False "
               "to accept the risk and remember to download the CSVs before closing.")
        if REQUIRE_PERSISTENT:
            raise RuntimeError(msg)
        print("[storage] WARNING -- " + msg)
        return os.path.abspath("/content/followup_outputs")

OUT = _resolve_out()
os.makedirs(OUT, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

SCALES = {
    "smoke": dict(dims=[2, 4, 8],        orders=[3],       seeds=[0, 1],
                  budgets=[4_000, 16_000],           grids=[4, 8, 16],
                  alphas=[0.0, 0.5, 1.0],            epochs=40,   n_base=2_000,
                  e2_rounds=4, e2_inner=40),
    "trim":  dict(dims=[2, 4, 8, 16],    orders=[2, 3],    seeds=[0, 1, 2, 3, 4],
                  budgets=[4_000, 16_000, 64_000],   grids=[4, 8, 16, 32],
                  alphas=[0.0, 0.25, 0.5, 0.75, 1.0], epochs=200, n_base=8_000,
                  e2_rounds=6, e2_inner=100),
    # 'full' is tuned for a single overnight session. Three seeds rather than five: the
    # trend and joint fits operate on (k, N, alpha) and (d, G) cells, so seeds only
    # stabilise the cell mean and the third seed already does most of that. The E1 grid
    # stops at 48 because G = 64 at d = 32 alone demands n_train = 592k, which is a
    # disproportionate share of the budget for one cell.
    "full":  dict(dims=[2, 4, 8, 16, 32], orders=[2, 3, 4], seeds=[0, 1, 2],
                  budgets=[2_000, 8_000, 32_000, 128_000],
                  grids=[4, 8, 16, 32, 48],
                  alphas=[0.0, 0.25, 0.5, 0.75, 1.0], epochs=500, n_base=16_000,
                  e2_rounds=8, e2_inner=150),
}
S = SCALES[SCALE]

# ---- Wall-clock budget ---------------------------------------------------------------
# Each experiment stops gracefully when its share is spent, so an interrupted session
# still leaves a complete, analysable subset rather than a crash. All three runners are
# resumable, so simply re-running the notebook continues where it stopped.
# RESUMPTION RUN (E3 completion).
# In the first full run E1 (450/450) and E2 (75/75) completed, while E3 was cut off by its
# time share at d = 16 and never reached d = 32. All three runners are resumable, so this
# configuration hands almost the whole budget to E3; E1 and E2 will find their work done
# and exit within seconds.
#
# For a run from scratch use MAX_HOURS = 10 and {"E3": 0.40, "E1": 0.50, "E2": 0.10}.
MAX_HOURS   = 6.0
TIME_SHARE  = {"E3": 0.90, "E1": 0.05, "E2": 0.05}

class Budget:
    'Graceful stop after a fixed share of the wall-clock budget.'
    def __init__(self, name):
        self.name = name
        self.limit = MAX_HOURS * 3600 * TIME_SHARE[name]
        self.t0 = time.time()
        self.stopped = False
    def elapsed(self): return time.time() - self.t0
    def exhausted(self):
        if self.elapsed() > self.limit and not self.stopped:
            self.stopped = True
            print(f"  [{self.name}] wall-clock share of "
                  f"{self.limit/3600:.1f} h reached -- stopping gracefully. "
                  f"Re-run the notebook to continue.")
        return self.stopped

CAP_RATIO   = 0.25       # E1: every cell must satisfy params / n_train <= this
BATCH       = 2_048
WEIGHT_DECAY= 1e-5
LR          = 3e-3
N_TEST_FRAC = 0.5        # test split size as a fraction of train

print(f"scale = {SCALE}   device = {DEVICE}")
print(f"outputs -> {OUT}")
if DEVICE.type == "cpu" and SCALE != "smoke":
    print("\n*** WARNING: no GPU detected and SCALE is not 'smoke'. This will take days. ***\n")
print(json.dumps({k: v for k, v in S.items()}, indent=2))

## 1. Shared primitives

The spline layer, the targets and the trainer, reproduced exactly as in the main pipeline so
that the two notebooks are comparable. If you change anything here, change it there too.

In [ ]:
class KANLayer(nn.Module):
    """B-spline KAN layer. Parameters per edge: (G + k) coefficients + 1 residual weight."""
    def __init__(self, in_dim, out_dim, grid=8, order=3, grid_range=(-1., 1.)):
        super().__init__()
        self.order, self.grid, self.grid_range = order, grid, grid_range
        g0, g1 = grid_range
        step = (g1 - g0) / grid
        self.register_buffer("knots", torch.arange(-order, grid + order + 1) * step + g0)
        self.coeff = nn.Parameter(torch.randn(in_dim, out_dim, grid + order) * 0.1)
        self.w_res = nn.Parameter(torch.randn(in_dim, out_dim) * 0.1)

    def _basis(self, x):
        g0, g1 = self.grid_range
        eps = 1e-6 * (g1 - g0)
        x = x.clamp(g0 + eps, g1 - eps).unsqueeze(-1)
        t = self.knots
        b = ((x >= t[:-1]) & (x < t[1:])).to(x.dtype)
        for k in range(1, self.order + 1):
            ld = (t[k:-1]   - t[:-(k + 1)]).clamp_min(1e-8)
            rd = (t[k + 1:] - t[1:-k]     ).clamp_min(1e-8)
            b = (x - t[:-(k + 1)]) / ld * b[..., :-1] + (t[k + 1:] - x) / rd * b[..., 1:]
        return b

    def forward(self, x):
        return (torch.einsum("bik,iok->bio", self._basis(x), self.coeff)
                + F.silu(x).unsqueeze(-1) * self.w_res).sum(1)


class MLP(nn.Module):
    def __init__(self, d, width, depth=2):
        super().__init__()
        self.inp = nn.Linear(d, width)
        self.blocks = nn.ModuleList([nn.ModuleList(
            [nn.LayerNorm(width), nn.Linear(width, width), nn.Linear(width, width)])
            for _ in range(depth)])
        self.head = nn.Linear(width, 1)
    def forward(self, x):
        h = self.inp(x)
        for ln, a, b in self.blocks:
            h = h + b(F.gelu(a(ln(h))))
        return self.head(h)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def relative_l2(pred, y):
    return (torch.linalg.norm(pred - y) / torch.linalg.norm(y).clamp_min(1e-12)).item()

class EdgeKAN(nn.Module):
    """Additive probe: one layer d -> 1.  E = d."""
    def __init__(self, d, grid, order):
        super().__init__(); self.kan = KANLayer(d, 1, grid=grid, order=order)
    def forward(self, x): return self.kan(x)


class KANet(nn.Module):
    """Two-layer Kolmogorov-Arnold probe, hidden width H = 2d+1.  E = (2d+1)(d+1)."""
    def __init__(self, d, grid, order):
        super().__init__()
        H = 2 * d + 1
        self.l1 = KANLayer(d, H, grid=grid, order=order)
        self.l2 = KANLayer(H, 1, grid=grid, order=order)
    def forward(self, x): return self.l2(self.l1(x))


def edge_count(kind, d):
    return d if kind == "edge_kan" else (2 * d + 1) * (d + 1)


def mlp_for_budget(d, budget, depth=2, wmax=2048):
    a, b = 2 * depth, 4 * depth + d + 2
    w0 = (-b + math.sqrt(max(b * b - 4 * a * (1 - budget), 0.0))) / (2 * a)
    cands = [w for w in {int(math.floor(w0)) - 1, int(math.floor(w0)),
                         int(math.ceil(w0)), int(math.ceil(w0)) + 1} if 2 <= w < wmax] or [2]
    return MLP(d, min(cands, key=lambda w: abs(a * w * w + b * w + 1 - budget)), depth)


# ---- targets ------------------------------------------------------------------------
def _additive(X):
    psi = lambda t: torch.sin(math.pi * t) + 0.3 * torch.cos(3 * math.pi * t)
    return psi(X).sum(1, keepdim=True) / math.sqrt(X.shape[1])

def _ka_struct(X, seed):
    d = X.shape[1]; H = 2 * d + 1
    g = torch.Generator().manual_seed(seed + 31)
    a  = torch.randn(H, d, generator=g)
    ph = torch.rand(H, d, generator=g) * 2 * math.pi
    c  = torch.randn(H, 1, generator=g) / math.sqrt(H)
    inner = (a.unsqueeze(0) * torch.sin(math.pi * X.unsqueeze(1) + ph.unsqueeze(0))).sum(-1)
    return torch.tanh(inner / math.sqrt(d)) @ c

def _interaction(X, seed):
    d = X.shape[1]
    g = torch.Generator().manual_seed(seed + 71)
    centres = torch.rand(8, d, generator=g) * 2 - 1
    w = torch.randn(8, 1, generator=g)
    d2 = ((X.unsqueeze(1) - centres.unsqueeze(0)) ** 2).sum(-1) / d
    return torch.exp(-1.5 * d2) @ w + 0.5 * X[:, :min(d, 4)].prod(1, keepdim=True)

def make_target(kind, d, n_tr, n_te, seed):
    g = torch.Generator().manual_seed(seed + 1000)
    Xtr = torch.rand(n_tr, d, generator=g) * 2 - 1
    Xte = torch.rand(n_te, d, generator=g) * 2 - 1
    if kind == "additive":
        fn = _additive
    elif kind == "kastruct":
        fn = lambda X: _ka_struct(X, seed)
    elif kind.startswith("interp"):
        al = float(kind.split("@")[1])
        fa, fi = _additive, lambda X: _interaction(X, seed)
        sa = fa(Xtr).std().clamp_min(1e-8); si = fi(Xtr).std().clamp_min(1e-8)
        fn = lambda X: (1 - al) * fa(X) / sa + al * fi(X) / si
    else:
        raise ValueError(kind)
    ytr, yte = fn(Xtr), fn(Xte)
    mu, sd = ytr.mean(), ytr.std().clamp_min(1e-8)
    to = lambda t: t.to(DEVICE, torch.float32)
    return to(Xtr), to((ytr - mu) / sd), to(Xte), to((yte - mu) / sd)


def train_eval(model, data, epochs, seed, lr=LR, batch=BATCH):
    """Plain AdamW + cosine. Deliberately the same protocol as the paper."""
    seed_everything(seed)
    Xtr, ytr, Xte, yte = data
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = Xtr.shape[0]
    g = torch.Generator(device="cpu").manual_seed(seed)
    model.train(); last = float("nan")
    for _ in range(epochs):
        perm = torch.randperm(n, generator=g).to(Xtr.device)
        tot, nb = 0.0, 0
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        sch.step(); last = tot / max(nb, 1)
    model.eval()
    with torch.no_grad():
        def chunked(X):
            return torch.cat([model(X[i:i + 8192]) for i in range(0, X.shape[0], 8192)])
        return dict(rel_l2=relative_l2(chunked(Xte), yte),
                    rel_l2_train=relative_l2(chunked(Xtr), ytr),
                    train_mse=last)

print("primitives ready")


def _load(csv, label):
    'Read a result file, returning an empty frame (not an exception) if it is missing.'
    if not os.path.exists(csv):
        print(f"[{label}] no results yet at {csv} -- this experiment has not produced any "
              f"cells. Re-run the notebook to continue it.")
        return pd.DataFrame()
    df = pd.read_csv(csv)
    if df.empty:
        print(f"[{label}] result file is empty.")
    return df


def calibrate(verbose=True):
    """Measure this machine's actual throughput and estimate the run time.

    Two regimes are timed because they behave differently: E3 trains thousands of tiny
    models where the bottleneck is per-step launch overhead, while E1 trains a few large
    ones where it is arithmetic.
    """
    def rate(d, G, k, n, epochs=3):
        data = make_target("kastruct", d, n, 512, 0)
        seed_everything(0); m = KANet(d, G, k)
        t0 = time.time(); train_eval(m, data, epochs, 0); dt = time.time() - t0
        return epochs * math.ceil(n / BATCH) / max(dt, 1e-9)

    r_small = rate(4, 8, 3, S["n_base"])          # E3-like
    r_large = rate(min(16, max(S["dims"])), 32, 3, 4 * S["n_base"])   # E1-like

    e3_cfg = len(list(itertools.product(S["orders"], S["dims"], S["budgets"],
                                        S["alphas"], S["seeds"])))
    e3_steps = e3_cfg * 2 * S["epochs"] * math.ceil(S["n_base"] / BATCH)
    e1_steps = sum(S["epochs"] * math.ceil(
        max(S["n_base"], math.ceil(((d if kd == "edge_kan" else (2*d+1)*(d+1))
                                    * (G + k + 1)) / CAP_RATIO)) / BATCH)
        for kd, k, d, G, _ in itertools.product(("kanet", "edge_kan"), S["orders"],
                                                S["dims"], S["grids"], S["seeds"]))
    e2_cells = len(S["orders"]) * len(S["dims"]) * len(S["grids"])
    e2_steps = e2_cells * (S["epochs"] + S["e2_rounds"] * S["e2_inner"]) * \
               math.ceil(S["n_base"] / BATCH)

    hours = (e3_steps / r_small + (e1_steps + e2_steps) / r_large) / 3600
    if verbose:
        print(f"throughput: {r_small:,.0f} steps/s (small models) | "
              f"{r_large:,.0f} steps/s (large)")
        print(f"work: E3 {e3_steps:,} + E1 {e1_steps:,} + E2 {e2_steps:,} steps")
        print(f"ESTIMATED RUN TIME: {hours:.1f} h   "
              f"(wall-clock budget is {MAX_HOURS:.0f} h)")
        if hours > MAX_HOURS:
            print(f"  -> over budget. The runners will stop gracefully and you can resume;")
            print(f"     or reduce seeds / grids in the config cell.")
    return hours


ESTIMATED_HOURS = calibrate()

## 2. Experiment E3 — re-specifying P2

**The defect.** P2 tested `FullKAN` against a benchmark of $2(k+1)$. But `FullKAN` is a
projection $\mathbb{R}^d\to\mathbb{R}^w$ followed by a $w\to w$ spline block, so its edge count
is $E = w^2$ with $w$ set by the budget. Solving the sizing quadratic gives an edge-count
elasticity between $-0.067$ and $-0.008$ across the swept dimensions: **the edge count of
`FullKAN` is effectively independent of $d$**, so the edge law predicts a dimensional slope
near zero for it, not $2(k+1)$.

**The fix.** Use a model whose edge count genuinely scales as $2d^2$: hold the hidden width at
$H = 2d+1$ and let the *grid* absorb the budget, $G = N/E - (k+1)$. Then $E \sim 2d^2$ holds by
construction and $2(k+1)$ becomes a legitimate benchmark.

The cell below first verifies the defect numerically, then runs the corrected comparison.

In [ ]:
def fullkan_edge_elasticity(dims, budgets, grid=8, order=3):
    """Demonstrates the defect: FullKAN's edge count barely moves with d."""
    rows = []
    for N in budgets:
        a, b = grid + order + 1, None
        Es = []
        for d in dims:
            b = d + 4
            w0 = (-b + math.sqrt(max(b * b - 4 * a * (1 - N), 0.0))) / (2 * a)
            cands = [w for w in {int(math.floor(w0)), int(math.ceil(w0))} if w >= 2] or [2]
            w = min(cands, key=lambda w: abs(a * w * w + b * w + 1 - N))
            Es.append(w * w)
        beta = float(np.polyfit(np.log(dims), np.log(Es), 1)[0])
        rows.append(dict(budget=N, E_at_dmin=Es[0], E_at_dmax=Es[-1],
                         dlogE_dlogd=round(beta, 4),
                         edge_law_prediction=round((order + 1) * beta, 3)))
    return pd.DataFrame(rows)


class KANetBudget(nn.Module):
    """Genuine two-layer K-A topology at a fixed parameter budget.

    Hidden width is pinned at H = 2d+1 by the superposition theorem, so E = (2d+1)(d+1)
    is fixed by d alone and the budget is absorbed by the grid:  G = N/E - (k+1).
    This is the architecture for which the 2(k+1) benchmark actually applies.
    """
    def __init__(self, d, budget, order, min_grid=2, max_grid=256):
        super().__init__()
        H = 2 * d + 1
        E = d * H + H
        G = int(round(budget / E - (order + 1)))
        self.grid = int(min(max(G, min_grid), max_grid))
        self.edges, self.H = E, H
        self.l1 = KANLayer(d, H, grid=self.grid, order=order)
        self.l2 = KANLayer(H, 1, grid=self.grid, order=order)
    def forward(self, x): return self.l2(self.l1(x))


print("The defect (Table 16 of the manuscript):")
print(fullkan_edge_elasticity(S["dims"], S["budgets"]).to_string(index=False))

print("\nThe corrected architecture -- realised edge count and grid:")
rows = []
for N in S["budgets"]:
    for d in S["dims"]:
        m = KANetBudget(d, N, S["orders"][0])
        rows.append(dict(budget=N, d=d, E=m.edges, G=m.grid, params=count_params(m)))
chk = pd.DataFrame(rows)
print(chk.pivot(index="d", columns="budget", values=["E", "G"]).to_string())
for N, g in chk.groupby("budget"):
    beta = float(np.polyfit(np.log(g.d), np.log(g.E), 1)[0])
    print(f"  budget {N}:  dlogE/dlogd = {beta:.3f}   (edge law now predicts "
          f"{(S['orders'][0]+1)*beta:.2f} at k={S['orders'][0]})")

In [ ]:
def run_E3(S, out_csv):
    """P2 re-specified: KANetBudget vs parameter-matched MLP on the mixture target."""
    done = set()
    if os.path.exists(out_csv):
        done = set(pd.read_csv(out_csv)["key"])
        print(f"resuming E3: {len(done)} cells present")
    rows, t0 = [], time.time()
    B = Budget("E3")
    # Order by dimension, ascending, so that an interrupted run still covers every
    # (N, alpha) cell at the dimensions it does reach. Because completed keys are skipped,
    # a resumption run advances straight to the dimensions that are still missing.
    jobs = sorted(itertools.product(S["orders"], S["dims"], S["budgets"],
                                    S["alphas"], S["seeds"]), key=lambda j: j[1])
    remaining = [j for j in jobs
                 if any(f"E3|{kd}|k{j[0]}|d{j[1]}|N{j[2]}|a{j[3]}|s{j[4]}" not in done
                        for kd in ("kanet_budget", "mlp"))]
    if len(remaining) < len(jobs):
        missing_dims = sorted({j[1] for j in remaining})
        print(f"E3: {len(jobs) - len(remaining)} configs already done; "
              f"{len(remaining)} remaining, at dimensions {missing_dims}")
    for i, (k, d, N, al, s) in enumerate(jobs):
        if B.exhausted(): break
        for kind in ("kanet_budget", "mlp"):
            key = f"E3|{kind}|k{k}|d{d}|N{N}|a{al}|s{s}"
            if key in done: continue
            n_tr = S["n_base"]
            data = make_target(f"interp@{al}", d, n_tr, int(n_tr * N_TEST_FRAC), s)
            seed_everything(s)
            m = KANetBudget(d, N, k) if kind == "kanet_budget" else mlp_for_budget(d, N)
            r = train_eval(m, data, S["epochs"], s)
            rows.append(dict(key=key, exp="E3", kind=kind, k=k, d=d, N=N, alpha=al, seed=s,
                             grid=getattr(m, "grid", None), edges=getattr(m, "edges", None),
                             params=count_params(m), **r))
        if rows and len(rows) % 40 == 0:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv))
            rows = []
            print(f"  {i+1}/{len(jobs)} configs, {(time.time()-t0)/60:.1f} min")
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                  header=not os.path.exists(out_csv))
    print(f"E3 done in {(time.time()-t0)/60:.1f} min")


E3_CSV = os.path.join(OUT, f"E3_p2_respecified_{SCALE}.csv")
run_E3(S, E3_CSV)

In [ ]:
def analyse_E3(csv, sat=None, n_boot=2000, min_dims=3):
    """Absorption gap against the CORRECT benchmark, plus the pre-registered trend test."""
    # The pre-registered filter is 0.8. At 'smoke' scale the models are barely trained,
    # so nothing survives it; the relaxed threshold there is a smoke-test convenience and
    # must NOT be used for any reported result.
    if sat is None:
        sat = 1.5 if SCALE == "smoke" else 0.8
    df = _load(csv, "E3")
    if df.empty: return pd.DataFrame(), None
    kb = df[df.kind == "kanet_budget"]
    agg = (kb.groupby(["k", "N", "alpha", "d"])["rel_l2"].mean().reset_index())
    print(f"E3: error range {agg.rel_l2.min():.3f} to {agg.rel_l2.max():.3f}; "
          f"saturation filter at {sat}"
          + ("  [RELAXED for smoke -- not reportable]" if sat != 0.8 else ""))

    rows = []
    for (k, N, al), g in agg.groupby(["k", "N", "alpha"]):
        g = g[(g.rel_l2 < sat) & (g.rel_l2 > 0)].sort_values("d")
        if g["d"].nunique() < min_dims: continue
        slope = float(np.polyfit(np.log(g.d), np.log(g.rel_l2), 1)[0])
        # benchmark from the REALISED edge elasticity of this architecture
        E = (2 * g.d.values + 1) * (g.d.values + 1)
        beta = float(np.polyfit(np.log(g.d.values), np.log(E), 1)[0])
        rows.append(dict(k=int(k), N=int(N), alpha=float(al),
                         measured_d_slope=round(slope, 3),
                         benchmark_asymptotic=2 * (k + 1),
                         benchmark_realised=round((k + 1) * beta, 3),
                         gap_asymptotic=round(2 * (k + 1) - slope, 3),
                         gap_realised=round((k + 1) * beta - slope, 3)))
    p2 = pd.DataFrame(rows)
    n_groups = agg.groupby(["k", "N", "alpha"]).ngroups
    print(f"E3: {n_groups} (k,N,alpha) groups; {len(p2)} survived the saturation filter "
          f"(err < {sat}) and the >= {min_dims} distinct dimensions requirement")
    if len(p2) < 3:
        print("  -> too few for a trend test. Expected at 'smoke' scale; use 'trim' or "
              "'full', or widen the dimension grid.")
        return p2, None

    MIN_TREND_POINTS = 6
    if len(p2) < MIN_TREND_POINTS:
        print(f"  -> only {len(p2)} points. A Spearman test on this many is not "
              f"interpretable: perfect monotonicity arises by chance and returns rho = 1, "
              f"p = 0. Refusing to report a trend. Increase the scale or widen the grid.")
        return p2, None

    out = {}
    for col in ("gap_asymptotic", "gap_realised"):
        rho, p = stats.spearmanr(p2.alpha, p2[col])
        rng = np.random.default_rng(0); sl = []
        for _ in range(n_boot):
            idx = rng.integers(0, len(p2), len(p2))
            if len(np.unique(p2.alpha.values[idx])) < 2: continue
            sl.append(np.polyfit(p2.alpha.values[idx], p2[col].values[idx], 1)[0])
        lo, hi = np.percentile(sl, [2.5, 97.5]) if sl else (np.nan, np.nan)
        out[col] = dict(rho=round(float(rho), 3), p=round(float(p), 4),
                        slope_lo=round(float(lo), 3), slope_hi=round(float(hi), 3),
                        supported=bool(rho < 0 and p < 0.05 and hi < 0))
    return p2, out


E3, E3_TREND = analyse_E3(E3_CSV)
print("E3 -- absorption gap by alpha, against both benchmarks:")
if len(E3):
    print(E3.groupby("alpha")[["measured_d_slope", "gap_asymptotic", "gap_realised"]]
            .mean().round(3).to_string())
print("\nPre-registered trend test (rho < 0, p < 0.05, slope CI entirely below 0):")
print(json.dumps(E3_TREND, indent=2) if E3_TREND else "n/a at this scale")

### 2b. Is the target getting easier with dimension?

The first `trim` run returned a **negative** dimensional slope for `KANetBudget` (about $-0.7$
to $-1.0$): error *falls* as the dimension grows. Neither the edge law nor any alternative
predicts that, so before interpreting E3 at all we must establish whether the fault lies in the
architecture or in the target.

The `MLP` arm of E3 settles it. Its capacity is set by the budget and is essentially
independent of $d$, so its dimensional slope measures the difficulty of the target itself.

* MLP slope **positive** — the target hardens with dimension, as it should. The negative
  `KANetBudget` slope is then an architectural effect worth investigating.
* MLP slope **negative or flat** — the target is *easier* in high dimension, and E3 measures
  nothing. The likely cause is the radial component of `_interaction`, whose Gaussian bumps
  flatten under concentration of measure as $d$ grows. In that case P2 in the manuscript is
  affected too, because it uses the same target family.

This diagnostic is why the notebook does not report E3 before running it.

In [ ]:
def target_difficulty(csv, min_dims=3):
    'Compare the dimensional slope of the fixed-capacity baseline against the probe.'
    df = _load(csv, "E3")
    if df.empty: return pd.DataFrame()
    out = []
    for kind, g in df.groupby("kind"):
        ag = g.groupby(["N", "alpha", "d"])["rel_l2"].mean().reset_index()
        slopes = []
        for (N, a), gg in ag.groupby(["N", "alpha"]):
            gg = gg.sort_values("d")
            if gg.d.nunique() >= min_dims and (gg.rel_l2 > 0).all():
                slopes.append(float(np.polyfit(np.log(gg.d), np.log(gg.rel_l2), 1)[0]))
        if slopes:
            out.append(dict(kind=kind, n_fits=len(slopes),
                            mean_d_slope=round(float(np.mean(slopes)), 3),
                            min_d_slope=round(float(np.min(slopes)), 3),
                            max_d_slope=round(float(np.max(slopes)), 3)))
    return pd.DataFrame(out)


DIFF = target_difficulty(E3_CSV)
TARGET_OK = None
if DIFF.empty:
    print("No E3 cells yet -- the target-difficulty diagnostic cannot run.")
else:
    print("Dimensional slope by architecture (MLP = target difficulty, "
          "capacity ~ independent of d):")
    print(DIFF.to_string(index=False))
    print("\nRaw MLP error levels, d x budget (should rise down each column):")
    _m = pd.read_csv(E3_CSV)
    print(_m[_m.kind == "mlp"].pivot_table(index="d", columns="N", values="rel_l2",
                                           aggfunc="mean").round(4).to_string())
    _mlp = DIFF[DIFF.kind == "mlp"]
    TARGET_OK = bool(len(_mlp) and _mlp.mean_d_slope.iloc[0] > 0.2)
    print("\nVERDICT:")
if TARGET_OK:
    print("  MLP slope is positive -- the target hardens with dimension, as intended.")
    print("  E3 is interpretable; a negative KANetBudget slope is an architectural effect.")
elif TARGET_OK is False:
    print("  MLP slope is NOT clearly positive -- the target does not harden with dimension.")
    print("  E3 measures nothing in its current form, AND the manuscript's P2 uses the same")
    print("  target family, so Section 5.3 needs revisiting. Do not report E3.")
    print("  Next step: replace the radial component of _interaction with one whose")
    print("  difficulty is dimension-stable, or renormalise it per dimension.")

## 3. Experiment E1 — decoupling capacity from sample size

**The defect.** The probe sweep held $n_{\mathrm{tr}} = 16{,}000$ fixed while sweeping capacity.
Consequently `KANet` is over-parametrised across much of its range — every cell at $d = 32$ has
$N/n_{\mathrm{tr}} > 1$ — while `EdgeKAN` never exceeds $0.136$. The two probes were not measured
in the same statistical regime, and the headline contrast is confounded with that difference.

**Why sub-setting the existing run does not fix it.** Refitting the two-layer law on cells with
$N/n_{\mathrm{tr}} < 0.5$ leaves the coefficient essentially unchanged, because that restriction
removes $d = 32$ entirely. Larger samples must actually be generated.

**The fix.** Tie the sample size to the budget: $n_{\mathrm{tr}} = \max(n_0, \lceil N/\rho\rceil)$
with $\rho = 0.25$, so every cell is approximation-limited by construction.

In [ ]:
def required_n(params, cap_ratio=CAP_RATIO, n_base=None):
    return int(max(n_base, math.ceil(params / cap_ratio)))


def run_E1(S, out_csv, kinds=("kanet", "edge_kan")):
    """Probe sweep with the sample size tied to the parameter budget."""
    done = set()
    if os.path.exists(out_csv):
        done = set(pd.read_csv(out_csv)["key"])
        print(f"resuming E1: {len(done)} cells present")
    rows, t0 = [], time.time()
    B = Budget("E1")
    # Sort by the sample size each cell demands, cheapest first. An interrupted run then
    # yields a coarse but COMPLETE grid over (d, G) rather than a few expensive corners.
    def _cost(j):
        kind, k, d, G, s = j
        return max(S["n_base"], math.ceil(edge_count(kind, d) * (G + k + 1) / CAP_RATIO))
    jobs = sorted(itertools.product(kinds, S["orders"], S["dims"], S["grids"], S["seeds"]),
                  key=_cost)
    print(f"E1: {len(jobs)} cells, n_train from {_cost(jobs[0]):,} to {_cost(jobs[-1]):,}")
    for i, (kind, k, d, G, s) in enumerate(jobs):
        if B.exhausted(): break
        key = f"E1|{kind}|k{k}|d{d}|G{G}|s{s}"
        if key in done: continue
        E = edge_count(kind, d)
        params = E * (G + k + 1)
        n_tr = required_n(params, n_base=S["n_base"])
        target = "additive" if kind == "edge_kan" else "kastruct"
        data = make_target(target, d, n_tr, int(n_tr * N_TEST_FRAC), s)
        seed_everything(s)
        m = EdgeKAN(d, G, k) if kind == "edge_kan" else KANet(d, G, k)
        r = train_eval(m, data, S["epochs"], s)
        rows.append(dict(key=key, exp="E1", kind=kind, target=target, k=k, d=d, grid=G,
                         seed=s, params=params, edges=E, n_train=n_tr,
                         cap_ratio=round(params / n_tr, 4), **r))
        if len(rows) % 25 == 0:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv))
            rows = []
            print(f"  {i+1}/{len(jobs)} cells, {(time.time()-t0)/60:.1f} min "
                  f"(largest n so far: {n_tr:,})")
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                  header=not os.path.exists(out_csv))
    print(f"E1 done in {(time.time()-t0)/60:.1f} min")


E1_CSV = os.path.join(OUT, f"E1_capacity_decoupled_{SCALE}.csv")
run_E1(S, E1_CSV)

In [ ]:
def joint_fit_both(cells, k):
    """(A) log eps ~ log N + log E   and   (B) log eps ~ log G_exact."""
    N = cells["params"].values.astype(float)
    E = cells["edges"].values.astype(float)
    y = np.log(cells["err"].values)
    XA = np.column_stack([np.ones(len(N)), np.log(N), np.log(E)])
    cA = np.linalg.lstsq(XA, y, rcond=None)[0]
    Gx = N / E - (k + 1); ok = Gx > 0
    cG = (np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), np.log(Gx[ok])]),
                          y[ok], rcond=None)[0][1] if ok.sum() >= 4 else np.nan)
    return float(cA[1]), float(cA[2]), float(cG)


def analyse_E1(csv, sat=0.6, n_boot=2000):
    df = _load(csv, "E1")
    if df.empty: return pd.DataFrame()
    rows = []
    for (kind, k), g in df.groupby(["kind", "k"]):
        cells = (g.groupby(["d", "grid", "params", "edges"])["rel_l2"]
                   .mean().reset_index().rename(columns={"rel_l2": "err"}))
        cells = cells[(cells.err < sat) & (cells.err > 0)]
        if len(cells) < 4 or cells["d"].nunique() < 2: continue
        cN, cE, cG = joint_fit_both(cells, k)
        rng = np.random.default_rng(0); bE, bG = [], []
        for _ in range(n_boot):
            f = joint_fit_both(cells.iloc[rng.integers(0, len(cells), len(cells))], k)
            bE.append(f[1])
            if np.isfinite(f[2]): bG.append(f[2])
        qE = np.percentile(bE, [2.5, 97.5]) if bE else [np.nan] * 2
        qG = np.percentile(bG, [2.5, 97.5]) if bG else [np.nan] * 2
        rows.append(dict(kind=kind, k=int(k), n_cells=len(cells),
                         max_cap_ratio=round(float(g.cap_ratio.max()), 3),
                         cE=round(cE, 3), cE_lo=round(qE[0], 3), cE_hi=round(qE[1], 3),
                         cE_pred=k + 1,
                         cE_pass=bool(abs(cE - (k + 1)) <= 0.25 * (k + 1)),
                         cG=round(cG, 3), cG_lo=round(qG[0], 3), cG_hi=round(qG[1], 3),
                         cG_pred=-(k + 1),
                         cG_pass=bool(abs(cG + (k + 1)) <= 0.15 * (k + 1))))
    return pd.DataFrame(rows)


def dslope_fixed_grid_E1(csv, k=None):
    """The premise test, re-run in the approximation-limited regime."""
    df = _load(csv, "E1")
    if df.empty: return pd.DataFrame()
    if k is not None: df = df[df.k == k]
    out = {}
    for kind, g in df.groupby("kind"):
        ag = g.groupby(["grid", "d"])["rel_l2"].mean().reset_index()
        row = {}
        for G, gg in ag.groupby("grid"):
            gg = gg.sort_values("d")
            if len(gg) >= 3 and (gg.rel_l2 > 0).all():
                row[int(G)] = round(float(np.polyfit(np.log(gg.d),
                                                     np.log(gg.rel_l2), 1)[0]), 2)
        out[kind] = row
    return pd.DataFrame(out).T


E1 = analyse_E1(E1_CSV)
print("E1 -- joint fit with capacity decoupled from sample size:")
print(E1.to_string(index=False))
print("\nE1 -- dimensional slope at fixed resolution (premise predicts 0):")
print(dslope_fixed_grid_E1(E1_CSV).to_string())
print("\nCompare against the paper: KANet gave c_E ~ -0.1 and c_G ~ +0.3 in the "
      "capacity-confounded regime.")

## 4. Experiment E2 — alternating exact solve

Given a **fixed** inner map, the outer layer is linear in its parameters, so its optimum is
available in closed form. Alternating between an exact outer solve and gradient steps on the
inner map gives a monotonically decreasing training error, hence a **certified attainable
error** for the two-layer family.

This is an upper bound on what is attainable, not a lower bound on what any trained model can
reach: a different inner map could do better. It cannot certify `KANet` the way
Proposition 2 certifies `EdgeKAN`. What it *can* do is show whether AdamW leaves a gap against
a demonstrably reachable solution — the question Section 5.2 leaves open for this architecture.

In [ ]:
def outer_exact_solve(model, Xtr, ytr, ridge=1e-8, dtype=torch.float64):
    """Solve the outer KAN layer exactly, with the inner map held fixed.

    Returns (train relative L2, condition number of the outer design matrix).
    """
    with torch.no_grad():
        Z = model.l1(Xtr)                                  # inner activations, fixed
        B = model.l2._basis(Z).reshape(Z.shape[0], -1)
        Phi = torch.cat([B, F.silu(Z)], 1).to(dtype)
        y = ytr.to(dtype)
        G = Phi.T @ Phi
        G = G + ridge * torch.eye(G.shape[0], dtype=dtype, device=G.device) * G.diagonal().mean()
        sol = torch.linalg.solve(G, Phi.T @ y)              # ridge-stabilised normal equations
        cond = float(torch.linalg.cond(Phi.to(torch.float64)))
        pred = (Phi @ sol).float()

        # write the solution back into the layer so training can continue from it
        n_basis = model.l2.grid + model.l2.order
        H = Z.shape[1]
        model.l2.coeff.copy_(sol[:H * n_basis].reshape(H, 1, n_basis).float())
        model.l2.w_res.copy_(sol[H * n_basis:].reshape(H, 1).float())
    return relative_l2(pred, ytr), cond


def alternating_solve(d, G, k, data, rounds=8, inner_steps=150, lr=LR, seed=0):
    """Block coordinate descent: exact outer solve, then gradient steps on the inner map."""
    seed_everything(seed)
    Xtr, ytr, Xte, yte = data
    model = KANet(d, G, k).to(DEVICE)
    history = []
    for r in range(rounds):
        tr_err, cond = outer_exact_solve(model, Xtr, ytr)
        # gradient steps on the INNER layer only
        for p in model.l2.parameters(): p.requires_grad_(False)
        opt = torch.optim.AdamW([p for p in model.l1.parameters()],
                                lr=lr, weight_decay=WEIGHT_DECAY)
        model.train()
        for _ in range(inner_steps):
            opt.zero_grad(set_to_none=True)
            F.mse_loss(model(Xtr), ytr).backward(); opt.step()
        for p in model.l2.parameters(): p.requires_grad_(True)
        history.append(dict(round=r, train_rel_l2=round(tr_err, 6), cond=f"{cond:.2e}"))
    tr_err, cond = outer_exact_solve(model, Xtr, ytr)       # final exact outer solve
    model.eval()
    with torch.no_grad():
        te_err = relative_l2(model(Xte), yte)
    history.append(dict(round=rounds, train_rel_l2=round(tr_err, 6), cond=f"{cond:.2e}"))
    return tr_err, te_err, pd.DataFrame(history)


def run_E2(S, out_csv, rounds=None, inner_steps=None):
    """Compare AdamW against the alternating certified-attainable solution."""
    rounds = rounds or S["e2_rounds"]; inner_steps = inner_steps or S["e2_inner"]
    done = set()
    if os.path.exists(out_csv):
        prev = pd.read_csv(out_csv)
        done = set(prev["key"]) if "key" in prev.columns else set()
        print(f"resuming E2: {len(done)} cells present")
    rows, t0 = [], time.time()
    B = Budget("E2")
    for k in S["orders"]:
        for d in S["dims"]:
            for G in S["grids"]:
                if B.exhausted(): break
                s = S["seeds"][0]
                key = f"E2|k{k}|d{d}|G{G}|s{s}"
                if key in done: continue
                n_tr = S["n_base"]
                data = make_target("kastruct", d, n_tr, int(n_tr * N_TEST_FRAC), s)

                seed_everything(s)
                base = train_eval(KANet(d, G, k), data, S["epochs"], s)

                alt_tr, alt_te, hist = alternating_solve(d, G, k, data, rounds,
                                                         inner_steps, seed=s)
                monotone = bool((np.diff(hist.train_rel_l2.values) <= 1e-9).all())
                rows.append(dict(key=key, exp="E2", k=k, d=d, grid=G, seed=s,
                                 adamw_train=round(base["rel_l2_train"], 6),
                                 adamw_test=round(base["rel_l2"], 6),
                                 alt_train=round(alt_tr, 6), alt_test=round(alt_te, 6),
                                 train_gap=round(base["rel_l2_train"] - alt_tr, 6),
                                 monotone=monotone,
                                 cond_final=hist.cond.iloc[-1]))
                print(f"  k={k} d={d} G={G}: AdamW train {base['rel_l2_train']:.4f} vs "
                      f"alternating {alt_tr:.4f}  (gap {base['rel_l2_train']-alt_tr:+.4f})")
                pd.DataFrame(rows[-1:]).to_csv(out_csv, mode="a", index=False,
                                               header=not os.path.exists(out_csv))
    print(f"E2 done in {(time.time()-t0)/60:.1f} min")
    return _load(out_csv, "E2")


E2_CSV = os.path.join(OUT, f"E2_alternating_solve_{SCALE}.csv")
E2 = run_E2(S, E2_CSV)
print("\nE2 summary:")
print(f"  alternating sequence monotone in {E2.monotone.mean()*100:.0f}% of cells")
print(f"  AdamW above the certified-attainable solution in "
      f"{(E2.train_gap > 0).mean()*100:.0f}% of cells; mean train gap "
      f"{E2.train_gap.mean():+.4f}")
print(E2.to_string(index=False))

## 5. What to put in the paper

The cell below writes a summary the manuscript can quote directly, and states plainly which
claims each experiment does and does not license.

In [ ]:
lines = [f"Follow-up experiments, scale = {SCALE}, device = {DEVICE}", ""]

# --- E3 ---
if len(E3):
    g0 = E3[E3.alpha == E3.alpha.min()].gap_realised.mean()
    g1 = E3[E3.alpha == E3.alpha.max()].gap_realised.mean()
    lines += ["E3 -- P2 re-specified against a genuine K-A topology:",
              f"  measured d-slope ranges {E3.measured_d_slope.min():.2f} to "
              f"{E3.measured_d_slope.max():.2f}",
              f"  gap against the realised benchmark: {g0:.2f} at alpha={E3.alpha.min()}, "
              f"{g1:.2f} at alpha={E3.alpha.max()}"]
    if E3_TREND:
        t = E3_TREND["gap_realised"]
        lines += [f"  trend test: rho = {t['rho']} (p = {t['p']}), slope CI "
                  f"[{t['slope_lo']}, {t['slope_hi']}] -> "
                  f"{'SUPPORTED' if t['supported'] else 'UNSUPPORTED'}",
                  "  P2 moves from 'untested' to tested. Report as pre-specified in the",
                  "  manuscript but executed after submission of the original study.", ""]

# --- E1 ---
if len(E1):
    lines += ["E1 -- capacity decoupled from sample size:",
              f"  every cell satisfies params/n_train <= {E1.max_cap_ratio.max():.3f}"]
    for r in E1.itertuples():
        lines.append(f"  {r.kind} k={r.k}: c_E {r.cE:+.3f} [{r.cE_lo:+.3f},{r.cE_hi:+.3f}] "
                     f"(pred {r.cE_pred}, pass={r.cE_pass}) | "
                     f"c_G {r.cG:+.3f} (pred {r.cG_pred}, pass={r.cG_pass})")
    kn = E1[E1.kind == "kanet"]
    if len(kn):
        lines += ["",
                  "  Compare with the paper: in the capacity-confounded regime KANet gave",
                  "  c_E ~ -0.1 and c_G ~ +0.3. If the sign has flipped here, the headline",
                  "  exponent has become measurable and the manuscript's central negative",
                  "  result is superseded -- Sections 4.4, 5.2 and 6 must be rewritten.", ""]

# --- E2 ---
if len(E2):
  lines += ["E2 -- alternating exact solve (certified ATTAINABLE, not a lower bound):",
          f"  NOTE: comparability requires AdamW to have a fair budget. At scale "
          f"'{SCALE}' it trains for {S['epochs']} epochs; at 'smoke' this is far too few and "
          f"the gap below is inflated. Only the 'full' figure is quotable.",
          f"  monotone descent in {E2.monotone.mean()*100:.0f}% of cells",
          f"  AdamW above the attainable solution in {(E2.train_gap > 0).mean()*100:.0f}% "
          f"of cells, mean train gap {E2.train_gap.mean():+.4f}",
          "  This does NOT extend Proposition 2 to the two-layer family. It shows only",
          "  whether AdamW leaves a gap against a demonstrably reachable solution.", ""]
else:
  lines += ["E2 -- no cells completed yet.", ""]

lines += ["Scope: none of these results may revise the pre-registered verdict of the",
          "original study. They extend it. E3 and E1 were named in Section 6.3 before",
          "being run; E2 is exploratory.", "",
          "=" * 70,
          "READ THIS FIRST", "=" * 70]

# --- the three questions to answer before touching the manuscript --------------------
_mlp = DIFF[DIFF.kind == "mlp"] if len(DIFF) else DIFF
_ok  = bool(len(_mlp) and _mlp.mean_d_slope.iloc[0] > 0.2)
lines += ["",
          f"Q1. Does the target harden with dimension?  "
          f"{'YES' if _ok else 'NO'}  (MLP d-slope "
          f"{_mlp.mean_d_slope.iloc[0] if len(_mlp) else float('nan'):+.2f})",
          "    If NO: E3 is uninterpretable and the manuscript's P2 (Section 5.3) rests on",
          "    the same target family, so it needs revisiting. Do not report E3."]

_kn = E1[E1.kind == "kanet"] if len(E1) else E1
if len(_kn):
    _cg = _kn.cG.mean()
    if _cg < -1.0:
        _msg = "MEASURABLE -- the headline exponent has become estimable"
    elif _cg < 0:
        _msg = "PARTIAL -- sign has flipped but the magnitude is short of -(k+1)"
    else:
        _msg = "UNCHANGED -- capacity was not the confound"
    lines += ["",
              f"Q2. Does decoupling capacity change the two-layer exponent?  {_msg}",
              f"    c_G = {_cg:+.3f} (paper: about +0.3);  c_E = {_kn.cE.mean():+.3f} "
              f"(paper: about -0.1)",
              "    MEASURABLE -> Sections 4.4, 5.2 and 6 must be rewritten.",
              "    PARTIAL    -> add a table and a paragraph; the negative result survives.",
              "    UNCHANGED  -> the negative result is STRENGTHENED; the most obvious",
              "                  alternative explanation has been eliminated."]

lines += ["",
          f"Q3. Is the alternating solve sound?  "
          + (f"{'YES' if E2.monotone.all() else 'NO -- check the ridge term'}  "
             f"(monotone in {E2.monotone.mean()*100:.0f}% of cells); "
             f"AdamW gap {E2.train_gap.mean():+.4f}" if len(E2) else "no cells yet")]

summary = "\n".join(lines)
print(summary)
with open(os.path.join(OUT, f"followup_summary_{SCALE}.txt"), "w") as f:
    f.write(summary)
print("\nwritten:", sorted(os.listdir(OUT)))